In [1]:
import pandas as pd
import duckdb

raw = pd.read_parquet("../data/raw/steam_games.parquet",
                      columns=["appID", "name", "release_date", "genres",
                               "developers", "positive", "negative"])

# same filters as the cleaning notebook, minus the 10-review minimum
raw = raw[~raw["name"].str.contains("playtest", case=False, na=False)]

def is_empty(x):
    return x is None or len(x) == 0
raw = raw[~raw["genres"].apply(is_empty) & ~raw["developers"].apply(is_empty)]

software_genres = {"Game Development", "Education", "Design & Illustration",
                   "Animation & Modeling", "Utilities", "Software Training",
                   "Video Production", "Audio Production", "Photo Editing",
                   "Web Publishing", "Accounting"}
raw = raw[~raw["genres"].apply(lambda g: all(x in software_genres for x in g))]

raw["release_date"] = pd.to_datetime(raw["release_date"], format="%b %d, %Y", errors="coerce")
raw["year"] = raw["release_date"].dt.year
raw["total_reviews"] = raw["positive"] + raw["negative"]

print(raw.shape)
print("Latest release date:", raw["release_date"].max())

(114360, 9)
Latest release date: 2026-09-08 00:00:00


In [2]:
query = """
SELECT
    year,
    COUNT(*)                                                         AS releases,
    SUM(CASE WHEN total_reviews >= 10 THEN 1 ELSE 0 END)             AS reached_10_reviews,
    ROUND(AVG(CASE WHEN total_reviews >= 10 THEN 1 ELSE 0 END) * 100, 1) AS pct_reached_10
FROM raw
WHERE year >= 2006
GROUP BY year
ORDER BY year
"""
duckdb.sql(query).df()

,year,releases,reached_10_reviews,pct_reached_10
0,2006,69,66.0,95.7
1,2007,90,90.0,100.0
2,2008,155,148.0,95.5
3,2009,305,288.0,94.4
4,2010,253,240.0,94.9
5,2011,258,256.0,99.2
6,2012,317,316.0,99.7
7,2013,435,433.0,99.5
8,2014,1485,1470.0,99.0
9,2015,2455,2377.0,96.8


In [3]:
raw[raw["name"] == "Schedule I"][["name", "release_date", "positive", "negative", "total_reviews"]]

,name,release_date,positive,negative,total_reviews
99453,Schedule I,2025-03-24,200803,3238,204041


In [5]:
recent = raw[raw["release_date"] >= "2024-01-01"]
monthly = recent.groupby(recent["release_date"].dt.to_period("M")).agg(
    releases=("appID", "count"),
    pct_with_any_review=("total_reviews", lambda s: round((s > 0).mean() * 100, 1)),
)
monthly

,releases,pct_with_any_review
release_date,,
2024-01,1167,70.2
2024-02,1407,71.1
2024-03,1339,71.4
2024-04,1325,71.4
2024-05,1528,69.3
2024-06,1378,66.9
2024-07,1526,67.7
2024-08,1601,66.5
2024-09,1444,69.0


**Data issue found:** review counts are complete for games released through May 2025 (59–71% of each month's releases have reviews) but drop to ~3–6% from June 2025 onward, and to 0% for 2026. The dataset's review data appears to have been captured earlier than its game list (which runs to early Feb 2026). Decision: release-volume analysis uses full years 2006–2025; review-based analysis uses games released through 2024. The cleaned dataset excludes games released after May 31, 2025.

In [6]:
# monthly release counts from ALL releases (raw), 2014-2024
rel = raw[(raw["year"] >= 2014) & (raw["year"] <= 2024)].copy()
rel["month"] = rel["release_date"].dt.to_period("M")

monthly = rel.groupby("month").size().rename("releases").reset_index()
monthly["year"] = monthly["month"].dt.year

# crowding = this month's releases / that year's average month
monthly["crowding"] = monthly["releases"] / monthly.groupby("year")["releases"].transform("mean")
monthly["crowding_band"] = pd.cut(
    monthly["crowding"],
    bins=[0, 0.9, 1.1, 10],
    labels=["1. Quieter (<90% of year avg)", "2. Typical (90-110%)", "3. Busier (>110%)"],
)

monthly["crowding_band"].value_counts().sort_index()

crowding_band
1. Quieter (<90% of year avg)    35
2. Typical (90-110%)             66
3. Busier (>110%)                31
Name: count, dtype: int64

In [7]:
clean = pd.read_parquet("../data/processed/steam_clean.parquet")
clean = clean[clean["release_date"].dt.year.between(2014, 2024)].copy()
clean["month"] = clean["release_date"].dt.to_period("M")

games = clean[["appID", "month", "pct_positive", "total_reviews"]].merge(
    monthly[["month", "crowding_band"]], on="month", how="left")
games["crowding_band"] = games["crowding_band"].astype(str)
games = games.drop(columns=["month"])   # DuckDB doesn't read pandas month periods

query = """
SELECT
    crowding_band,
    COUNT(*)                                          AS games,
    ROUND(QUANTILE_CONT(pct_positive, 0.25) * 100, 1) AS p25,
    ROUND(MEDIAN(pct_positive) * 100, 1)              AS median,
    ROUND(QUANTILE_CONT(pct_positive, 0.75) * 100, 1) AS p75,
    MEDIAN(total_reviews)                             AS median_reviews
FROM games
GROUP BY crowding_band
ORDER BY crowding_band
"""
duckdb.sql(query).df()

,crowding_band,games,p25,median,p75,median_reviews
0,1. Quieter (<90% of year avg),10518,65.2,79.3,89.7,68.5
1,2. Typical (90-110%),28224,68.2,81.8,91.7,49.0
2,3. Busier (>110%),12194,65.0,79.6,90.0,67.0


In [8]:
rel2 = rel.merge(monthly[["month", "crowding_band"]], on="month")
rel2.groupby("crowding_band", observed=True).agg(
    releases=("appID", "count"),
    pct_reached_10=("total_reviews", lambda s: round((s >= 10).mean() * 100, 1)),
)

,releases,pct_reached_10
crowding_band,,
1. Quieter (<90% of year avg),16702,63.0
2. Typical (90-110%),53013,53.2
3. Busier (>110%),19588,62.3


In [9]:
monthly["cal_month"] = monthly["month"].dt.month
monthly.groupby("cal_month")["crowding"].mean().round(2)

cal_month
1     0.76
2     0.86
3     0.93
4     0.96
5     1.05
6     0.89
7     1.05
8     1.07
9     1.07
10    1.19
11    1.10
12    1.08
Name: crowding, dtype: float64

In [10]:
pd.crosstab(monthly["year"], monthly["crowding_band"])

crowding_band,1. Quieter (<90% of year avg),2. Typical (90-110%),3. Busier (>110%)
year,,,
2014,5,3,4
2015,4,3,5
2016,5,2,5
2017,6,1,5
2018,2,9,1
2019,1,9,2
2020,3,7,2
2021,1,9,2
2022,4,6,2


In [11]:
# reception and reach, each relative to the game's own release year
g = clean[["appID", "release_date", "pct_positive", "total_reviews"]].copy()
g["year"] = g["release_date"].dt.year
g["month"] = g["release_date"].dt.to_period("M")
g = g.merge(monthly[["month", "crowding_band"]], on="month")

g["pos_vs_year"] = (g["pct_positive"] - g.groupby("year")["pct_positive"].transform("median")) * 100
g["reviews_vs_year"] = g["total_reviews"] / g.groupby("year")["total_reviews"].transform("median")

# share reaching 10 reviews, relative to the release year's average (all releases)
rel2["reached"] = rel2["total_reviews"] >= 10
rel2["reached_vs_year"] = (rel2["reached"] - rel2.groupby("year")["reached"].transform("mean")) * 100

print(g.groupby("crowding_band", observed=True).agg(
    games=("appID", "count"),
    pos_pts_vs_year=("pos_vs_year", "median"),
    reviews_x_vs_year=("reviews_vs_year", "median"),
).round(2))

print(rel2.groupby("crowding_band", observed=True)["reached_vs_year"].mean().round(1))

                               games  pos_pts_vs_year  reviews_x_vs_year
crowding_band                                                           
1. Quieter (<90% of year avg)  10518            -0.59               1.07
2. Typical (90-110%)           28224             0.00               1.00
3. Busier (>110%)              12194             0.38               0.93
crowding_band
1. Quieter (<90% of year avg)    2.6
2. Typical (90-110%)            -0.2
3. Busier (>110%)               -1.6
Name: reached_vs_year, dtype: float64


**Finding (release crowding):** Steam releases grew ~40x from 2013 to 2025. Within each year, January is the quietest month (~76% of an average month) and October the busiest (~119%). A first comparison suggested quiet and busy months both outperformed typical months, but this was an artifact of release year: 2014–2017 had mostly quiet/busy months and older games have had more time to collect reviews. After comparing each game only to others released the same year, crowding shows no association with reception (within ±1 point) and a small association with reach: games from busier months received ~7% fewer reviews and were ~4 points less likely to reach 10 reviews than games from quieter months. Crowding and seasonality cannot be fully separated in this data.